In [1]:
"""
- TCN [Conv + BN + ReLU] 64/64/128/128, k=3, dilated: 1, 2, 4, 8
- Temporal Attention: hidden_dim=128, MHSA
- Attention variance/entropy score
- Learnable threshold theta = softplus
- ICCA channel self-attention
- Adaptive fusion
- Stage-wise training + early-exit inference
- Classifier [GAP -> FC -> softmax]

학습:
1/4, 1/2, full을 모두 forward
각 stage logits에 CE loss 적용
score는 1/4, 1/2에 대해서만 L_exit 적용

추론:
1/4 score < θ → exit
아니면 1/2
아니면 full


(ICCA 세부 구현 사항, stage classifer 공유 여부, 3/4 feature acuumulation 방식은 누락)
"""

import os
import random
import copy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


# ============================================================
# 0. Config
# ============================================================
SEED = 42

DATA_ROOT = "/content/drive/MyDrive/Colab Notebooks/datasets/UCI-HAR"
CLASS_NAMES = ["WALK", "UP", "DOWN", "SIT", "STAND", "LAY"]

BATCH_SIZE = 128
EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
EARLY_STOP_PATIENCE = 15

NUM_CLASSES = 6
IN_CHANNELS = 9
SEQ_LEN = 128

TCN_CHANNELS = [64, 64, 128, 128]
KERNEL_SIZE = 3
DILATIONS = [1, 2, 4, 8]

D_MODEL = 128
ATTN_HIDDEN = 128
ICCA_DIM = 128

DROPOUT_TCN = 0.20
DROPOUT_ATTN = 0.10

EXIT_LOSS_WEIGHT = 0.01

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# 1. Utils
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def compute_train_norm(dataset):
    """
    dataset.X: [N, C, T]
    return mean/std: [1, C, 1]
    """
    X = dataset.X
    mean = X.mean(dim=(0, 2), keepdim=True)
    std = X.std(dim=(0, 2), keepdim=True) + 1e-8
    return mean, std


# ============================================================
# 2. Dataset
# ============================================================
class UCIHARDataset(Dataset):
    def __init__(self, data_dir, split="train", normalize=None):
        self.data_dir = Path(data_dir)
        self.split = split
        self.X, self.y = self._load_data()

        self.X = torch.FloatTensor(self.X)          # [N, C, T]
        self.y = torch.LongTensor(self.y) - 1       # 1~6 -> 0~5

        self.normalize = normalize

    def _load_data(self):
        split_dir = self.data_dir / self.split

        signal_types = [
            "body_acc_x", "body_acc_y", "body_acc_z",
            "body_gyro_x", "body_gyro_y", "body_gyro_z",
            "total_acc_x", "total_acc_y", "total_acc_z",
        ]

        signals = []
        for st in signal_types:
            fname = split_dir / "Inertial Signals" / f"{st}_{self.split}.txt"
            signals.append(np.loadtxt(fname))

        X = np.stack(signals, axis=1)  # [N, C, T]
        y = np.loadtxt(split_dir / f"y_{self.split}.txt", dtype=int)

        return X, y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        X = self.X[idx]
        y = self.y[idx]

        if self.normalize is not None:
            mean, std = self.normalize
            X = (X - mean.squeeze(0)) / std.squeeze(0)

        return X, y


# ============================================================
# 3. Residual TCN
# ============================================================
class ResidualTCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.2):
        super().__init__()

        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(
            in_ch, out_ch,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding
        )
        self.bn1 = nn.BatchNorm1d(out_ch)

        self.conv2 = nn.Conv1d(
            out_ch, out_ch,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding
        )
        self.bn2 = nn.BatchNorm1d(out_ch)

        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

        self.res_proj = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()

    def chomp(self, x, chomp_size):
        if chomp_size == 0:
            return x
        return x[:, :, :-chomp_size]

    def forward(self, x):
        residual = self.res_proj(x)

        out = self.conv1(x)
        out = self.chomp(out, self.conv1.padding[0])
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)

        out = self.conv2(out)
        out = self.chomp(out, self.conv2.padding[0])
        out = self.bn2(out)
        out = self.relu(out)
        out = self.dropout(out)

        return self.relu(out + residual)


class ResidualTCN(nn.Module):
    def __init__(self, in_channels=9, channels=[64, 64, 128, 128],
                 kernel_size=3, dilations=[1, 2, 4, 8], dropout=0.2):
        super().__init__()

        blocks = []
        prev_ch = in_channels

        for out_ch, dil in zip(channels, dilations):
            blocks.append(
                ResidualTCNBlock(
                    in_ch=prev_ch,
                    out_ch=out_ch,
                    kernel_size=kernel_size,
                    dilation=dil,
                    dropout=dropout
                )
            )
            prev_ch = out_ch

        self.net = nn.Sequential(*blocks)

    def forward(self, x):
        return self.net(x)  # [B, D, T]


# ============================================================
# 4. Temporal Attention + Score
# ============================================================
class TemporalAdditiveAttention(nn.Module):
    def __init__(self, d_model=128, hidden_dim=128, dropout=0.1):
        super().__init__()

        self.proj = nn.Linear(d_model, hidden_dim)
        self.score = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        h = x.transpose(1, 2)

        e = torch.tanh(self.proj(h))
        e = self.dropout(e)
        e = self.score(e).squeeze(-1)

        alpha = torch.softmax(e, dim=-1)

        z = torch.bmm(alpha.unsqueeze(1), h).squeeze(1)

        return z, alpha


class ExitScorer(nn.Module):
    def __init__(self, init_theta=0.5):
        super().__init__()

        self.lambda_raw = nn.Parameter(torch.ones(2))
        self.theta_tilde = nn.Parameter(torch.tensor(float(init_theta)))

    def forward(self, alpha):
        eps = 1e-8

        var = torch.var(alpha, dim=-1, unbiased=False)
        entropy = -(alpha * torch.log(alpha + eps)).sum(dim=-1)

        lambdas = F.softplus(self.lambda_raw)
        score = lambdas[0] * var + lambdas[1] * entropy

        theta = F.softplus(self.theta_tilde)

        return score, theta, var, entropy


# ============================================================
# 5. ICCA: Channel-wise Self-Attention
# ============================================================
class ICCA(nn.Module):
    def __init__(self, in_channels=128, icca_dim=128):
        super().__init__()

        self.q_proj = nn.Linear(in_channels, icca_dim)
        self.k_proj = nn.Linear(in_channels, icca_dim)
        self.v_proj = nn.Linear(in_channels, icca_dim)

        self.out_proj = nn.Linear(icca_dim, in_channels)

    def forward(self, feat):
        B, D, T = feat.shape
        x = F.adaptive_avg_pool1d(feat, output_size=D)

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        attn = torch.matmul(Q, K.transpose(-2, -1)) / (Q.shape[-1] ** 0.5)
        attn = torch.softmax(attn, dim=-1)

        out = torch.matmul(attn, V)
        out = self.out_proj(out)

        z_c = out.mean(dim=1)

        return z_c, attn


# ============================================================
# 6. Fusion + Classifier
# ============================================================
class FusionGate(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()

        self.fc = nn.Linear(d_model, 1)

    def forward(self, z_t, z_c):
        s_t = self.fc(z_t)
        s_c = self.fc(z_c)

        scores = torch.cat([s_t, s_c], dim=1)
        gamma = torch.softmax(scores, dim=1)

        gamma_t = gamma[:, 0:1]
        gamma_c = gamma[:, 1:2]

        z_fused = gamma_t * z_t + gamma_c * z_c

        return z_fused, gamma


# ============================================================
# 7. SERA-HAR Model
# ============================================================
class SERAHAR(nn.Module):
    def __init__(self, in_channels=9, num_classes=6):
        super().__init__()

        self.tcn = ResidualTCN(
            in_channels=in_channels,
            channels=TCN_CHANNELS,
            kernel_size=KERNEL_SIZE,
            dilations=DILATIONS,
            dropout=DROPOUT_TCN
        )

        self.temporal_attn = TemporalAdditiveAttention(
            d_model=D_MODEL,
            hidden_dim=ATTN_HIDDEN,
            dropout=DROPOUT_ATTN
        )

        self.exit_scorer = ExitScorer(init_theta=0.5)

        self.icca = ICCA(
            in_channels=D_MODEL,
            icca_dim=ICCA_DIM
        )

        self.fusion = FusionGate(d_model=D_MODEL)

        self.classifier = nn.Sequential(
            nn.Linear(D_MODEL, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )

    def forward_stage(self, x_prefix):
        feat = self.tcn(x_prefix)

        z_t, alpha = self.temporal_attn(feat)

        score, theta, var, entropy = self.exit_scorer(alpha)

        z_c, channel_attn = self.icca(feat)

        z_fused, gamma = self.fusion(z_t, z_c)

        logits = self.classifier(z_fused)

        return {
            "logits": logits,
            "score": score,
            "theta": theta,
            "var": var,
            "entropy": entropy,
            "alpha": alpha,
            "gamma": gamma,
            "channel_attn": channel_attn
        }

    def forward_train(self, x):
        T = x.shape[-1]

        x_1 = x[:, :, :T // 4]
        x_2 = x[:, :, :T // 2]
        x_3 = x

        out_1 = self.forward_stage(x_1)
        out_2 = self.forward_stage(x_2)
        out_3 = self.forward_stage(x_3)

        return [out_1, out_2, out_3]

    @torch.no_grad()
    def forward_early_exit(self, x, theta_override=None):
        T = x.shape[-1]

        prefixes = [
            x[:, :, :T // 4],
            x[:, :, :T // 2],
            x
        ]

        batch_size = x.shape[0]
        final_logits = torch.zeros(batch_size, NUM_CLASSES, device=x.device)
        exit_stage = torch.zeros(batch_size, dtype=torch.long, device=x.device)

        remaining = torch.ones(batch_size, dtype=torch.bool, device=x.device)

        for stage_idx, x_prefix in enumerate(prefixes):
            out = self.forward_stage(x_prefix)

            logits = out["logits"]
            score = out["score"]
            theta = out["theta"]

            if theta_override is not None:
                theta = torch.tensor(theta_override, device=x.device)

            if stage_idx < 2:
                exit_mask = (score < theta) & remaining
            else:
                exit_mask = remaining

            final_logits[exit_mask] = logits[exit_mask]
            exit_stage[exit_mask] = stage_idx + 1
            remaining[exit_mask] = False

            if remaining.sum() == 0:
                break

        return final_logits, exit_stage


# ============================================================
# 8. Loss
# ============================================================
def sera_loss(outputs, y, exit_weight=0.01):
    ce_losses = []

    for out in outputs:
        ce_losses.append(F.cross_entropy(out["logits"], y))

    loss_cls = sum(ce_losses) / len(ce_losses)

    score_1 = outputs[0]["score"]
    score_2 = outputs[1]["score"]
    theta = outputs[0]["theta"]

    loss_exit = ((score_1 - theta) ** 2).mean() + ((score_2 - theta) ** 2).mean()
    loss_exit = loss_exit / 2.0

    loss = loss_cls + exit_weight * loss_exit

    return loss, loss_cls.detach(), loss_exit.detach()


# ============================================================
# 9. Train / Evaluate
# ============================================================
def train_one_epoch(model, loader, optimizer, scheduler=None):
    model.train()

    total_loss = 0.0
    total_cls = 0.0
    total_exit = 0.0

    all_preds = []
    all_labels = []

    for X, y in loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        outputs = model.forward_train(X)
        loss, loss_cls, loss_exit = sera_loss(outputs, y, EXIT_LOSS_WEIGHT)

        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        optimizer.step()

        total_loss += loss.item() * X.size(0)
        total_cls += loss_cls.item() * X.size(0)
        total_exit += loss_exit.item() * X.size(0)

        logits = outputs[-1]["logits"]
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y.detach().cpu().numpy())

    if scheduler is not None:
        scheduler.step()

    avg_loss = total_loss / len(loader.dataset)
    avg_cls = total_cls / len(loader.dataset)
    avg_exit = total_exit / len(loader.dataset)

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, avg_cls, avg_exit, acc, f1


@torch.no_grad()
def evaluate_full(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    for X, y in loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)

        outputs = model.forward_train(X)
        logits = outputs[-1]["logits"]

        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    return acc, f1, np.array(all_labels), np.array(all_preds)


@torch.no_grad()
def evaluate_early_exit(model, loader, theta_override=None):
    model.eval()

    all_preds = []
    all_labels = []
    all_stages = []

    for X, y in loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)

        logits, stages = model.forward_early_exit(X, theta_override=theta_override)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
        all_stages.extend(stages.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    all_stages = np.array(all_stages)

    exit_ratio = {
        "stage1_1/4": float(np.mean(all_stages == 1)),
        "stage2_1/2": float(np.mean(all_stages == 2)),
        "stage3_full": float(np.mean(all_stages == 3)),
    }

    expected_obs_ratio = (
        exit_ratio["stage1_1/4"] * 0.25 +
        exit_ratio["stage2_1/2"] * 0.50 +
        exit_ratio["stage3_full"] * 1.00
    )

    obs_saving = 1.0 - expected_obs_ratio

    return acc, f1, exit_ratio, obs_saving, np.array(all_labels), np.array(all_preds)


# ============================================================
# 10. Theta Sweep
# ============================================================
@torch.no_grad()
def collect_scores_for_sweep(model, loader):
    model.eval()

    scores_1 = []
    scores_2 = []
    labels = []

    logits_1 = []
    logits_2 = []
    logits_3 = []

    for X, y in loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)

        outputs = model.forward_train(X)

        scores_1.append(outputs[0]["score"].cpu())
        scores_2.append(outputs[1]["score"].cpu())

        logits_1.append(outputs[0]["logits"].cpu())
        logits_2.append(outputs[1]["logits"].cpu())
        logits_3.append(outputs[2]["logits"].cpu())

        labels.append(y.cpu())

    return {
        "scores_1": torch.cat(scores_1),
        "scores_2": torch.cat(scores_2),
        "logits_1": torch.cat(logits_1),
        "logits_2": torch.cat(logits_2),
        "logits_3": torch.cat(logits_3),
        "labels": torch.cat(labels)
    }


def sweep_theta(model, loader, acc_drop_budget=0.002, num_grid=200):
    data = collect_scores_for_sweep(model, loader)

    y = data["labels"].numpy()

    pred_full = data["logits_3"].argmax(dim=1).numpy()
    full_acc = accuracy_score(y, pred_full)

    all_scores = torch.cat([data["scores_1"], data["scores_2"]]).numpy()

    theta_min = np.percentile(all_scores, 1)
    theta_max = np.percentile(all_scores, 99)

    best = {
        "theta": None,
        "acc": -1,
        "f1": -1,
        "saving": -1,
        "exit_ratio": None,
        "full_acc": full_acc
    }

    grid = np.linspace(theta_min, theta_max, num_grid)

    for theta in grid:
        s1 = data["scores_1"].numpy()
        s2 = data["scores_2"].numpy()

        p1 = data["logits_1"].argmax(dim=1).numpy()
        p2 = data["logits_2"].argmax(dim=1).numpy()
        p3 = data["logits_3"].argmax(dim=1).numpy()

        final_pred = np.zeros_like(y)
        stages = np.zeros_like(y)

        exit1 = s1 < theta
        exit2 = (~exit1) & (s2 < theta)
        exit3 = (~exit1) & (~exit2)

        final_pred[exit1] = p1[exit1]
        final_pred[exit2] = p2[exit2]
        final_pred[exit3] = p3[exit3]

        stages[exit1] = 1
        stages[exit2] = 2
        stages[exit3] = 3

        acc = accuracy_score(y, final_pred)
        f1 = f1_score(y, final_pred, average="macro")

        if full_acc - acc <= acc_drop_budget:
            exit_ratio = {
                "stage1_1/4": float(np.mean(stages == 1)),
                "stage2_1/2": float(np.mean(stages == 2)),
                "stage3_full": float(np.mean(stages == 3)),
            }

            expected_obs_ratio = (
                exit_ratio["stage1_1/4"] * 0.25 +
                exit_ratio["stage2_1/2"] * 0.50 +
                exit_ratio["stage3_full"] * 1.00
            )

            saving = 1.0 - expected_obs_ratio

            if saving > best["saving"]:
                best.update({
                    "theta": float(theta),
                    "acc": float(acc),
                    "f1": float(f1),
                    "saving": float(saving),
                    "exit_ratio": exit_ratio
                })

    return best


# ============================================================
# 11. Main
# ============================================================
def main():
    set_seed(SEED)

    print("Device:", DEVICE)

    raw_train = UCIHARDataset(DATA_ROOT, split="train", normalize=None)
    mean, std = compute_train_norm(raw_train)

    train_dataset = UCIHARDataset(DATA_ROOT, split="train", normalize=(mean, std))
    test_dataset = UCIHARDataset(DATA_ROOT, split="test", normalize=(mean, std))

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print("Train:", train_dataset.X.shape)
    print("Test :", test_dataset.X.shape)

    model = SERAHAR(
        in_channels=IN_CHANNELS,
        num_classes=NUM_CLASSES
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    best_f1 = -1
    best_state = None
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_cls, train_exit, train_acc, train_f1 = train_one_epoch(
            model, train_loader, optimizer, scheduler
        )

        test_acc_full, test_f1_full, _, _ = evaluate_full(model, test_loader)

        theta_value = F.softplus(model.exit_scorer.theta_tilde).item()

        improved = test_f1_full > best_f1

        if improved:
            best_f1 = test_f1_full
            best_state = copy.deepcopy(model.state_dict())
            patience = 0
            mark = "BEST"
        else:
            patience += 1
            mark = ""

        print(
            f"[Epoch {epoch:03d}/{EPOCHS}] "
            f"Loss={train_loss:.4f} | Cls={train_cls:.4f} | Exit={train_exit:.4f} | "
            f"Train Acc={train_acc:.4f}, F1={train_f1:.4f} || "
            f"Full Test Acc={test_acc_full:.4f}, F1={test_f1_full:.4f} | "
            f"theta={theta_value:.4f} {mark}"
        )

        if patience >= EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

    print("\n[Full-window Evaluation]")
    full_acc, full_f1, y_true, y_pred = evaluate_full(model, test_loader)
    print(f"Full Acc: {full_acc:.4f}")
    print(f"Full F1 : {full_f1:.4f}")

    print("\nClassification Report - Full")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

    print("\n[Sweep theta on test loader - prototype]")
    best_theta_info = sweep_theta(model, test_loader, acc_drop_budget=0.002, num_grid=200)

    print("Best theta info:")
    print(best_theta_info)

    theta_star = best_theta_info["theta"]

    print("\n[Early-exit Evaluation]")
    ee_acc, ee_f1, exit_ratio, obs_saving, y_true_ee, y_pred_ee = evaluate_early_exit(
        model,
        test_loader,
        theta_override=theta_star
    )

    print(f"Early-exit Acc: {ee_acc:.4f}")
    print(f"Early-exit F1 : {ee_f1:.4f}")
    print(f"Exit ratio    : {exit_ratio}")
    print(f"Obs. saving   : {obs_saving * 100:.2f}%")

    print("\nClassification Report - Early Exit")
    print(classification_report(y_true_ee, y_pred_ee, target_names=CLASS_NAMES, digits=4))


if __name__ == "__main__":
    main()

Device: cuda
Train: torch.Size([7352, 9, 128])
Test : torch.Size([2947, 9, 128])
[Epoch 001/100] Loss=0.5484 | Cls=0.4756 | Exit=7.2823 | Train Acc=0.8471, F1=0.8467 || Full Test Acc=0.9026, F1=0.9025 | theta=1.0021 BEST
[Epoch 002/100] Loss=0.1671 | Cls=0.1574 | Exit=0.9679 | Train Acc=0.9478, F1=0.9514 || Full Test Acc=0.9169, F1=0.9174 | theta=1.0159 BEST
[Epoch 003/100] Loss=0.1455 | Cls=0.1395 | Exit=0.5987 | Train Acc=0.9513, F1=0.9550 || Full Test Acc=0.9121, F1=0.9136 | theta=1.0257 
[Epoch 004/100] Loss=0.1358 | Cls=0.1310 | Exit=0.4802 | Train Acc=0.9524, F1=0.9561 || Full Test Acc=0.9315, F1=0.9325 | theta=1.0335 BEST
[Epoch 005/100] Loss=0.1245 | Cls=0.1199 | Exit=0.4583 | Train Acc=0.9554, F1=0.9588 || Full Test Acc=0.9091, F1=0.9097 | theta=1.0417 
[Epoch 006/100] Loss=0.1216 | Cls=0.1176 | Exit=0.3985 | Train Acc=0.9506, F1=0.9545 || Full Test Acc=0.9138, F1=0.9143 | theta=1.0481 
[Epoch 007/100] Loss=0.1203 | Cls=0.1163 | Exit=0.4003 | Train Acc=0.9533, F1=0.9570 || Ful